# Segundo estudio de simulación: Matérn LMC

Este notebook deja solo el flujo principal. Las funciones viven en `src/codispersion_bootstrap/`.

## 0. Instalación local en VS Code

Ejecuta esta celda una vez si aparece `ModuleNotFoundError`. Después reinicia el kernel y continúa con el notebook.

In [ ]:
from pathlib import Path
import sys
import subprocess

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
ROOT = None
for path in candidates:
    if (path / 'pyproject.toml').exists() and (path / 'src' / 'codispersion_bootstrap').exists():
        ROOT = path
        break

if ROOT is None:
    raise RuntimeError(f'No encontré la carpeta raíz del proyecto. Estoy parado en: {cwd}')

print('Carpeta raíz encontrada:', ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])
print('Paquete instalado correctamente. Reinicia el kernel antes de seguir.')

## 1. Imports

In [1]:
import pandas as pd

from codispersion_bootstrap import MaternExperimentConfig, run_min_experiment_matern
from codispersion_bootstrap.experiments import save_results_json

## 2. Configuración del escenario

In [2]:
cfg = MaternExperimentConfig(
    n1=256,
    n2=256,
    rho0=0.75,
    range_pix=7.0,
    nu=1.0,
    anis_mode='aniso',
    angle_deg=45.0,
    anis_ratio=2.0,
    hmax=2,
    b=57,
    B=200,   # usar 200--800+ en corrida final
    seed=2025,
)
cfg

MaternExperimentConfig(n1=256, n2=256, rho0=0.75, range_pix=7.0, nu=1.0, anis_mode='aniso', angle_deg=45.0, anis_ratio=2.0, hmax=2, b=57, B=200, seed=2025, use_tukey=True, k_px=8, sampler='CBB', mode_rho='toroidal')

## 3. Ejecutar experimento

In [3]:
result = run_min_experiment_matern(cfg)
result.keys()

dict_keys(['params', 'H', 'rho_hat', 'boot'])

## 4. Tabla de codispersión por lag

In [4]:
rho_df = (
    pd.DataFrame.from_dict(result['rho_hat'], orient='index', columns=['rho_hat'])
    .rename_axis('h')
    .reset_index()
)
rho_df

,h,rho_hat
0,"(1, 0)",0.745957
1,"(0, 1)",0.743433
2,"(1, 1)",0.748233
3,"(-1, 1)",0.743971
4,"(2, 0)",0.746618
5,"(0, 2)",0.742361
6,"(2, 2)",0.747677
7,"(-2, 2)",0.744992


## 5. Tabla bootstrap

In [5]:
boot_df = (
    pd.DataFrame.from_dict(result['boot'], orient='index')
    .rename_axis('h')
    .reset_index()
)
boot_df

,h,var_boot,ci_lo,ci_hi
0,"(1, 0)",0.000037,0.734401,0.757610
1,"(0, 1)",0.000038,0.731134,0.755644
2,"(1, 1)",0.000067,0.729525,0.761733
3,"(-1, 1)",0.000031,0.733289,0.754724
4,"(2, 0)",0.000034,0.735919,0.757708
5,"(0, 2)",0.000038,0.730538,0.754146
6,"(2, 2)",0.000056,0.731144,0.759836
7,"(-2, 2)",0.000051,0.730279,0.758295


## 6. Guardar resultados

In [6]:
save_results_json(result, '../results/segundo_estudio_matern_resultados.json')